# 07 - AI Smart Config

> **When to use**: When you don't want to hand-write YAML configs and want AI to auto-analyze table structure and generate optimal configs.
>
> **Core concept**: The `sqlseed-ai` plugin uses an LLM to analyze schema semantics, auto-generating configs with a self-correction loop.

## Applicable Scenarios

- Complex table structure, don't want to hand-write config → AI auto-generates
- Unsure which generator to use → AI recommends based on column name semantics
- Need rapid prototyping → AI generates config with one command

## What You Will Learn

- SchemaAnalyzer workflow
- AiConfigRefiner self-correction loop
- Auto model selection and fallback
- ErrorSummary error classification
- Caching mechanism

⚠️ Requires API Key (see `.env.example`)

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| **→ 07** | **AI Smart Config** | **Plugins: AI** | **01** |
| 08 | MCP Server Integration | Plugins: MCP | 07 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---

In [1]:
# Prerequisite: pip install -e ".[dev,all]"
import sqlseed
from sqlseed import connect, fill, fill_from_config, preview

# Demo database setup
import sys
sys.path.insert(0, "..")  # for build_demo_db only
from build_demo_db import build
db_path = build()  # Force rebuild to ensure idempotent run

# Populate base dependencies
with connect(str(db_path)) as orch:
    orch.fill_table("organizations", count=5, seed=42)
    orch.fill_table("members", count=20, seed=42)
    orch.fill_table("projects", count=10, seed=42)
    orch.fill_table("tags", count=8, seed=42)

print(f"sqlseed {sqlseed.__version__} | Database: {db_path}")

c:\Users\14435\Desktop\sqlseed\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating organizations: 100%|██████████| 5/5 [00:00<00:00, 176.33it/s]
2026-06-13T05:48:26.052662Z [warning  ] AI API key not configured. Set GOOGLE_API_KEY, SQLSEED_AI_API_KEY, or OPENAI_API_KEY environment variable. For Ollama, set SQLSEED_AI_BACKEND=ollama.
Generating members: 100%|██████████| 20/20 [00:00<00:00, 279.26it/s]
2026-06-13T05:48:26.180159Z [warning  ] AI API key not configured. Set GOOGLE_API_KEY, SQLSEED_AI_API_KEY, or OPENAI_API_KEY environment variable. For Ollama, set SQLSEED_AI_BACKEND=ollama.
Generating tags: 100%|██████████| 8/8 [00:00<00:00, 142.17it/s]


sqlseed 0.2.2.dev1+g7f73669bf.d20260609 | Database: C:\Users\14435\Desktop\sqlseed\examples\sqlseed_demo.db


### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Schema Analysis | `plugins/sqlseed-ai/src/sqlseed_ai/analyzer.py` | `SchemaAnalyzer` |

> Corresponding architecture diagram: [§7 AI Plugin Architecture](../docs/architecture.zh-CN.md#7-ai-插件架构)

## 1. See It in Action — AI Auto-Analyzes Table Structure

Give sqlseed-ai a table name and it will auto-analyze the schema and generate the optimal config — **no hand-written YAML needed**:

```
Input: organizations table
Output: auto-generated YAML config (including generator, params, constraints)
```

Below we first verify the plugin installation, then demo the full workflow.

## 2. sqlseed-ai Installation Verification

In [2]:
try:
    import sqlseed_ai  # noqa: F401
    print("✅ sqlseed-ai installed")
except ImportError:
    print("⚠️ sqlseed-ai not installed")
    print("   Install: pip install -e ./plugins/sqlseed-ai")

✅ sqlseed-ai installed


## 3. SchemaAnalyzer Workflow

SchemaAnalyzer collects the full table context (columns, indexes, FKs, sample data), sends it to the LLM for analysis, and returns a YAML config.

In [3]:
from sqlseed_ai.analyzer import SchemaAnalyzer

from sqlseed import connect

with connect(str(db_path)) as orch:
    schema_ctx = orch.get_schema_context('organizations')
    print('Schema context keys:', list(schema_ctx.keys()))
    print(f'Columns: {len(schema_ctx["columns"])}')
    print(f'Foreign keys: {len(schema_ctx["foreign_keys"])}')
    print(f'Indexes: {len(schema_ctx["indexes"])}')
    print(f'Sample data: {len(schema_ctx["sample_data"])} rows')
    print(f'All tables: {schema_ctx["all_table_names"]}')

Schema context keys: ['table_name', 'columns', 'foreign_keys', 'indexes', 'sample_data', 'all_table_names', 'distribution']
Columns: 7
Foreign keys: 1
Indexes: 1
Sample data: 5 rows
All tables: ['organizations', 'members', 'sqlite_sequence', 'projects', 'tasks', 'reviews', 'tags', 'task_tags', 'attachments']


## 4. AiConfigRefiner Self-Correction Loop

AiConfigRefiner implements a closed loop: generate → validate → fix → re-validate. If no API Key is set, sample output is shown.

In [4]:
ORG_PATTERN = r'ORG-\d{4}'
from sqlseed_ai.config import AIConfig, AIBackend

# Use AIConfig.from_env() to auto-detect backend (supports LM Studio / Ollama / Google AI Studio)
# Env vars: SQLSEED_AI_BACKEND, SQLSEED_AI_BASE_URL, SQLSEED_AI_MODEL
ai_config = AIConfig.from_env()

# Check backend availability (LM Studio/Ollama don't need API Key; cloud backends do)
has_backend = ai_config.backend in (AIBackend.LM_STUDIO, AIBackend.OLLAMA) or ai_config.has_real_api_key

if has_backend:
    from sqlseed_ai.analyzer import SchemaAnalyzer
    from sqlseed_ai.refiner import AiConfigRefiner

    print(f'Backend: {ai_config.backend.value}')
    print(f'Model: {ai_config.resolve_model()}')
    try:
        print(f'Base URL: {ai_config.resolve_base_url()}')
    except ValueError as e:
        print(f'Config error: {e}')
    print()

    analyzer = SchemaAnalyzer(config=ai_config)
    refiner = AiConfigRefiner(analyzer, db_path=str(db_path))

    result = refiner.generate_and_refine('organizations')
    if isinstance(result, dict):
        print(f'AI generated config for {len(result)} columns:')
        for col_name, col_config in result.items():
            if isinstance(col_config, dict):
                print(f'  {col_name}: generator={col_config.get("generator", "N/A")}, params={col_config.get("params", {})}')  # noqa: E501
            else:
                print(f'  {col_name}: {col_config}')
    else:
        print('AI generated config (raw):')
        print(str(result)[:500])
else:
    print('No available AI backend detected. Set one of the following env vars:')
    print('  - SQLSEED_AI_API_KEY (Google AI Studio / OpenRouter)')
    print('  - SQLSEED_AI_BACKEND=lm_studio (LM Studio local inference)')
    print('  - SQLSEED_AI_BACKEND=ollama (Ollama local inference)')
    print()
    print('Showing sample AI output:')
    print()
    example_config = {
        'org_code': {'generator': 'pattern', 'params': {'pattern': ORG_PATTERN}},
        'name': {'generator': 'company'},
        'parent_code': {'generator': 'pattern', 'params': {'pattern': ORG_PATTERN}},
        'description': {'generator': 'sentence', 'params': {'nb_words': 8}},
        'is_active': {'generator': 'boolean'},
        'member_count': {'generator': 'integer', 'params': {'min_value': 1, 'max_value': 500}},
    }
    for col_name, col_config in example_config.items():
        print(f'  {col_name}: generator={col_config["generator"]}, params={col_config.get("params", {})}')



AI generated config for 3 columns:
  name: organizations
  count: 1000
  columns: [{'name': 'org_code', 'generator': 'pattern', 'params': {'regex': 'ORG-[0-9]{4,6}'}}, {'name': 'name', 'generator': 'company'}, {'name': 'parent_code', 'generator': 'foreign_key', 'params': {'ref_table': 'organizations', 'ref_column': 'org_code'}}, {'name': 'description', 'generator': 'text', 'params': {'min_length': 50, 'max_length': 200}}, {'name': 'created_at', 'generator': 'datetime', 'params': {'start_year': 2000, 'end_year': 2025}}]


## 5. Auto Model Selection

`select_best_model` tries free models by priority, with auto-fallback.

In [5]:
from sqlseed_ai._model_selector import _GEMMA_MODEL_PRIORITY

print(f'Gemma 4 model priority ({len(_GEMMA_MODEL_PRIORITY)} models):')
for i, model in enumerate(_GEMMA_MODEL_PRIORITY, 1):
    print(f'  {i}. {model.value} ({model.display_name})')

print('\nselect_gemma_model() tries by priority, with auto-fallback')



Gemma 4 model priority (5 models):
  1. gemma-4-31b-it (Gemma 4 31B Dense)
  2. gemma-4-26b-a4b-it (Gemma 4 26B A4B MoE (Recommended))
  3. gemma-4-12b-it (Gemma 4 12B Unified (Laptop))
  4. gemma-4-e4b-it (Gemma 4 E4B (4B Effective, Edge))
  5. gemma-4-e2b-it (Gemma 4 E2B (2B Effective, Edge))

select_gemma_model() tries by priority, with auto-fallback


## 6. ErrorSummary Error Classification

`summarize_error` classifies exceptions into 7 types, used in the AI self-correction loop.

In [6]:
from sqlseed_ai.errors import summarize_error

test_errors = [
    ValueError('count must be greater than 0'),
    TypeError("'NoneType' object is not iterable"),
    KeyError('missing_column'),
    ImportError('No module named faker'),
    FileNotFoundError('/nonexistent.db'),
    PermissionError('read-only database'),
    RuntimeError('unexpected error'),
]

print('ErrorSummary 7 error classifications:')
for err in test_errors:
    summary = summarize_error(err)
    print(f'  {type(err).__name__:20s} -> {summary.error_type}')



ErrorSummary 7 error classifications:
  ValueError           -> runtime_error
  TypeError            -> runtime_error
  KeyError             -> runtime_error
  ImportError          -> runtime_error
  FileNotFoundError    -> fatal
  PermissionError      -> fatal
  RuntimeError         -> runtime_error


## 7. End-to-End Workflow

Full flow: schema analysis → AI generates config → validate → fill data.

In [7]:
ORG_PATTERN = r'ORG-\d{4}'
from pathlib import Path

from sqlseed import fill_from_config
from sqlseed.config.loader import save_config
from sqlseed.config.models import ColumnConfig, GeneratorConfig, TableConfig

# Simulate AI-generated config (or use actual AI if key available)
ai_config = GeneratorConfig(
    db_path=str(db_path),
    tables=[
        TableConfig(
            name='organizations',
            count=3,
            clear_before=True,
            columns=[
                ColumnConfig(name='org_code', generator='pattern', params={'pattern': ORG_PATTERN}),
                ColumnConfig(name='name', generator='company'),
                ColumnConfig(name='description', generator='sentence'),
            ]
        )
    ]
)

# Save and fill
config_path = Path('_ai_demo_config.yaml')
save_config(ai_config, str(config_path))
print('Generated YAML config:')
print(config_path.read_text()[:300])

results = fill_from_config(str(config_path), clear_before=True)
for r in results:
    print(f'\nFilled {r.table_name}: {r.count} rows in {r.elapsed:.3f}s')

config_path.unlink(missing_ok=True)



Generated YAML config:
db_path: /Users/sunbo/Documents/webblock/sqlseed/examples/sqlseed_demo.db
provider: mimesis
locale: en_US
tables:
- name: organizations
  count: 3
  batch_size: 5000
  columns:
  - name: org_code
    generator: pattern
    provider: null
    params:
      pattern: ORG-\d{4}
    null_ratio: 0.0
    d


Generating organizations:   0%|          | 0/3 [00:00<?, ?it/s]


Filled organizations: 3 rows in 0.032s


## Summary

| Feature | Description |
|------|------|
| SchemaAnalyzer | Collects table context, sends to LLM |
| AiConfigRefiner | Self-correction loop: generate→validate→fix |
| Model Selection | 12 free models, auto-fallback |
| ErrorSummary | 7 error classifications |
| File Cache | Platform-standard cache dir |

**Next**: [08-mcp-server.ipynb](08-mcp-server.ipynb) — MCP Server Integration

In [8]:
# ✅ Validation: ensure data was successfully generated and written
import sqlite3
conn = sqlite3.connect(str(db_path))
try:
    # Basic row count validation
    member_count = conn.execute("SELECT COUNT(*) FROM members").fetchone()[0]
    assert member_count > 0, f"Expected members > 0, got {member_count}"
    print("✅ All assertions passed")
finally:
    conn.close()

✅ All assertions passed
